In [1]:
from lark import Lark, Tree
import os
from builder_ast import ASTBuilder

test_folder = "tests"
success_folder = test_folder + "/success"
fail_folder = test_folder + "/fails"

# Point to the main parser file
parser = Lark.open("parser.lark", parser="lalr")

In [2]:
minimal_program = success_folder + "/minimal.k"
with open(minimal_program, "r") as file:
    code = file.read()

tree = parser.parse(code)
print(tree.pretty())

start
  code
    program
    main
    {
    declarations
    procedures
    block
      begin
      ;
      statements
      end
      ;
    }



In [3]:
# tests 
# get .k files from the tests folder and parse them to see if the parser works correctly.



for filename in os.listdir(success_folder):
    if filename.endswith(".k"):
        with open(os.path.join(success_folder, filename), "r") as file:
            code = file.read()
            tree = parser.parse(code)
            print(f"Parsed {filename} successfully.")


Parsed while.k successfully.
Parsed variableDeclaration.k successfully.
Parsed expressions.k successfully.
Parsed block.k successfully.
Parsed writeConstant.k successfully.
Parsed variableIncrementDecrement copy.k successfully.
Parsed minimal.k successfully.
Parsed variableAssigment.k successfully.
Parsed if.k successfully.
Parsed procedure.k successfully.
Parsed declarations.k successfully.
Parsed for.k successfully.
Parsed variableWrite.k successfully.
Parsed if_else.k successfully.
Parsed simple_program.k successfully.


In [4]:
# abstract syntax tree

simple_program = success_folder + "/simple_program.k"
with open(simple_program, "r") as file:
    code = file.read()

tree = parser.parse(code)
print(tree.pretty())

start
  code
    program
    main
    {
    declarations
      declaration
        var
        arr
        ,
        i
        ,
        a
        ,
        b
        ,
        c
        :
        type	int
        ;
      declaration
        var
        x
        ,
        y
        :
        type	float
        ;
      declaration
        var
        ch
        :
        type	char
        ;
      declaration
        var
        flag
        :
        type	bool
        ;
    procedures
      procedure
        procedure
        assign_array_value
        {
        block
          begin
          ;
          statements
            statement
              action
                assignment
                  arr
                  :=
                  expression
                    boolean_expression
                      addition_expression
                        product_expression
                          value_expression	i
                          *
                          value_expre

In [5]:
# Tree exploration

tree.children[0]  # program main

declarations = tree.children[0].children[3] # declarations
procedures = tree.children[0].children[4] # procedures
block = tree.children[0].children[5].children[2] # begin-end block

In [6]:
declarations = declarations.children[0]

type = declarations.children[-2].children[0].value # type
raw_variables = [
    child
    for child in declarations.children
    if isinstance(child, Tree) and child.data.value == "name"
] # raw variables
type

'int'

In [7]:
variables = [
    (
        var.children[0].value, 
        len(var.children) > 2 and var.children[2] or None
     ) # (name, length?) where length is an expression
    for var in raw_variables
] # parsed variables
variables

[]

In [8]:
procedure_name = procedures.children[0].children[1].value # procedure assign_array_value
procedure_block = procedures.children[0].children[3].children[2].children # procedure block
procedure_block

[Tree(Token('RULE', 'statement'), [Tree(Token('RULE', 'action'), [Tree(Token('RULE', 'assignment'), [Token('ID', 'arr'), Token('ASSIGN', ':='), Tree(Token('RULE', 'expression'), [Tree(Token('RULE', 'boolean_expression'), [Tree(Token('RULE', 'addition_expression'), [Tree(Token('RULE', 'product_expression'), [Tree(Token('RULE', 'value_expression'), [Token('ID', 'i')]), Token('TIMES', '*'), Tree(Token('RULE', 'value_expression'), [Token('CTE', '2')])])])])])])]), Token('SEMICOLON', ';')])]

In [9]:
some_statement = block.children[0]
some_statement

Tree(Token('RULE', 'statement'), [Tree(Token('RULE', 'action'), [Tree(Token('RULE', 'write'), [Token('WRITE', 'write'), Token('LPAREN', '('), Tree(Token('RULE', 'expression'), [Tree(Token('RULE', 'boolean_expression'), [Tree(Token('RULE', 'addition_expression'), [Tree(Token('RULE', 'product_expression'), [Tree(Token('RULE', 'value_expression'), [Token('NOT', '!'), Token('CTE', 'true')])])])]), Token('OR', 'or'), Tree(Token('RULE', 'boolean_expression'), [Tree(Token('RULE', 'addition_expression'), [Tree(Token('RULE', 'product_expression'), [Tree(Token('RULE', 'value_expression'), [Token('LPAREN', '('), Tree(Token('RULE', 'expression'), [Tree(Token('RULE', 'boolean_expression'), [Tree(Token('RULE', 'addition_expression'), [Tree(Token('RULE', 'product_expression'), [Tree(Token('RULE', 'value_expression'), [Token('NOT', '!'), Token('CTE', 'false')])])])]), Token('AND', 'and'), Tree(Token('RULE', 'boolean_expression'), [Tree(Token('RULE', 'addition_expression'), [Tree(Token('RULE', 'product

In [10]:
ast = ASTBuilder().transform(tree)

ast.variables, ast.procedures, ast.block

([VarDeclNode(name='arr', var_type='int'),
  VarDeclNode(name='i', var_type='int'),
  VarDeclNode(name='a', var_type='int'),
  VarDeclNode(name='b', var_type='int'),
  VarDeclNode(name='c', var_type='int'),
  VarDeclNode(name='x', var_type='float'),
  VarDeclNode(name='y', var_type='float'),
  VarDeclNode(name='ch', var_type='char'),
  VarDeclNode(name='flag', var_type='bool')],
 [ProcedureNode(name=Token('PROCEDURE', 'procedure'), block=BlockNode(statements=[AssignmentNode(variable=IdentifierNode(name='arr'), expression=BinaryOpNode(left=IdentifierNode(name='i'), operator='*', right=IntegerNode(value=2)))])),
  ProcedureNode(name=Token('PROCEDURE', 'procedure'), block=BlockNode(statements=[WriteNode(expression=IdentifierNode(name='arr'))]))],
 BlockNode(statements=[WriteNode(expression=BinaryOpNode(left=UnaryOpNode(operator='!', operand=BoolNode(value=True)), operator='or', right=BinaryOpNode(left=UnaryOpNode(operator='!', operand=BoolNode(value=False)), operator='and', right=BoolNode